# Updating data from GenBank

Author: Alexander Maksiaev

Purpose: Update labels from previously gotten data from GISAID + Andersen, using Genbank.

In [1]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Combinations/GISAID_Andersen/B3_13_D1_1/"
temp_files = downloads + "Andersen/"

update_date = "05-12-2025"

os.chdir(originals)

## Collection Dates

In [2]:
# Upload saved data 
os.chdir(temp_files + "saved/")
metadata_genbank = pd.read_csv("metadata_genbank_4-22-2025.csv") # Since 1/1/2024
os.chdir(originals)

display(metadata_genbank)

,Unnamed: 0,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,...,Genotype,date,File Name,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,name_state,Collection_Date_Specific
0,0,SRR32804537,WGS,148.60,157337234,PRJNA1207547,SAMN47505727,Viral,55371987,USDA-NVSL,...,D1.1,2025-04-08_16-17-04,SRR32804537.fa,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.87%, 99.34%, 99.49%, 99.35%, 99.80%, 98.85%...","3, 15, 9, 11, 3, 11, 1, 7",Ran on FASTA - No Coverage Report,unknown,2025
1,1,SRR32804540,WGS,146.83,125026759,PRJNA1207547,SAMN47505724,Viral,45700705,USDA-NVSL,...,D1.1,2025-04-08_16-17-07,SRR32804540.fa,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.69%, 99.39%, 99.83%, 99.41%, 99.67%, 99.42%...","7, 11, 3, 10, 5, 6, 0, 10",Ran on FASTA - No Coverage Report,unknown,2025
2,2,SRR32804544,WGS,145.13,75849086,PRJNA1207547,SAMN47505721,Viral,27735348,USDA-NVSL,...,D1.1,2025-04-08_16-17-10,SRR32804544.fa,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.69%, 99.50%, 99.60%, 99.65%, 99.60%, 99.05%...","7, 9, 7, 6, 6, 10, 2, 10",Ran on FASTA - No Coverage Report,unknown,2025
3,3,SRR32804545,WGS,148.43,284460923,PRJNA1207547,SAMN47505720,Viral,98766118,USDA-NVSL,...,D1.1,2025-04-08_16-17-11,SRR32804545.fa,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.87%, 99.39%, 99.77%, 99.41%, 99.67%, 99.11%...","3, 11, 5, 10, 5, 10, 0, 9",Ran on FASTA - No Coverage Report,unknown,2025
4,4,SRR32804546,WGS,148.88,213319278,PRJNA1207547,SAMN47505719,Viral,74543339,USDA-NVSL,...,D1.1,2025-04-08_16-17-12,SRR32804546.fa,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.83%, 99.40%, 99.83%, 99.47%, 99.73%, 99.42%...","4, 11, 3, 9, 4, 6, 0, 9",Ran on FASTA - No Coverage Report,unknown,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,298,SRR33029747,WGS,147.87,245673050,PRJNA980729,SAMN47843702,Viral,85653750,USDA-NVSL,...,D1.1,2025-04-11_06-46-48,SRR33029747.fa,"PB1:ea3, MP:ea3, NA:am4N1, NS:ea3, NP:am13, PA...","ea3:22-013001-001:PB1, ea3:22-013001-001:MP, a...","99.45%, 99.80%, 99.42%, 98.93%, 99.60%, 98.56%...","10, 2, 6, 9, 6, 31, 8, 9",Ran on FASTA - No Coverage Report,unknown,2025
299,299,SRR33029749,WGS,147.88,167615693,PRJNA980729,SAMN47843700,Viral,58998925,USDA-NVSL,...,D1.1,2025-04-11_06-46-48,SRR33029749.fa,"PA:am4, NA:am4N1, HA:ea3, NS:ea3, PB1:ea3, MP:...","am4:24-030039-001:PA, am4N1:24-030039-001:NA, ...","98.37%, 99.42%, 99.53%, 98.93%, 99.39%, 99.80%...","35, 6, 8, 9, 11, 2, 9, 6",Ran on FASTA - No Coverage Report,unknown,2025
300,300,SRR33029750,WGS,146.73,95329644,PRJNA980729,SAMN47843699,Viral,33874309,USDA-NVSL,...,D1.1,2025-04-11_06-46-48,SRR33029750.fa,"HA:ea3, NA:am4N1, NS:ea3, PB1:ea3, NP:am13, MP...","ea3:22-013001-001:HA, am4N1:24-030039-001:NA, ...","99.53%, 99.42%, 98.93%, 99.44%, 99.60%, 99.80%...","8, 6, 9, 10, 6, 2, 9, 9",Ran on FASTA - No Coverage Report,unknown,2025
301,301,SRR33029751,WGS,146.12,269328456,PRJNA980729,SAMN47843690,Viral,99122459,USDA-NVSL,...,D1.1,2025-04-11_06-46-49,SRR33029751.fa,"PA:am4, HA:ea3, NP:am13, MP:ea3, PB2:am24, PB1...","am4:24-030039-001:PA, ea3:22-013001-001:HA, am...","99.30%, 99.06%, 99.73%, 100.00%, 99.74%, 99.12...","15, 16, 4, 0, 6, 20, 10, 7",Ran on FASTA - No Coverage Report,unknown,2025


In [3]:
# no_updates = pd.DataFrame()
# no_updates_isolate = []

# Get only labels that have no states or collection dates, and update them

def update(file_name, update_date):
    updates = {}
    with open(file_name) as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if line[0] == ">": # It's a header
                header = line 
                collection_date = header.split("|")[-3]
                state = header.split("/")[2]
                state = state.replace(": ", "-")
                geo_location_collection_date = state + "|" + collection_date
                isolate = header.split("/")[3]
                sequence = lines[i + 1] # Sequence always comes in one line after header
                # if "-" not in collection_date: # If there are no dashes, i.e. if it's just the year
                    # Find the correct collection date, if it exists
                row = metadata_genbank[metadata_genbank["isolate"] == isolate]
                
                try: # Isolate may not be in this dataset
                    id = row["BioSample"].values[0]
                    geo_location_collection_date = search_collection_date(id, row) # Update unknown dates, if possible
                except:
                    print("No BioSample nor date nor state found for isolate", isolate)
                        # no_updates_isolate.append(isolate)

                # if state == "USA": # If we don't have a state
                #     row = metadata_genbank[metadata_genbank["isolate"] == isolate]

                #     try: # Isolate may not be in this dataset
                #         genbank_name = row["genbank_name"].values[0]
                #         state = genbank_name.split("/")[2]
                #     except:
                #         print("No state found for isolate", isolate)
                #         # no_updates_isolate.append(isolate)

                updates[header] = [geo_location_collection_date, sequence]

        f.close()

    updates_df = pd.DataFrame.from_dict(updates, orient="index", columns=["geo_location_collection_date", "sequence"])
    updates_df["header"] = updates_df.index
    updates_df = updates_df.reset_index()

    updated_file_name = ".".join(file_name.split(".")[:-1]) + "_" + update_date + "_update." + file_name.split(".")[-1]

    with open(updated_file_name, "w") as g:

        for i, row in updates_df.iterrows():
            header = row["header"]
            # print(header)
            # print(header.split("|")[-3])
            
            header = header.replace(str(header.split("|")[-3]), str(row["geo_location_collection_date"])) # Only the first instance is replaced

            g.write(header)
            g.write(row["sequence"])

        g.close()

    # no_updates["isolate"] = no_updates_isolate
    # no_updates.to_csv("not_updated.csv")


In [4]:
# Create files with updates

os.chdir(originals)

for dirpath, dirs, files in os.walk(originals + "04-14-2025--05-14-2025_D1_1/"): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file)
        update(file_name, update_date)
    break 



No BioSample nor date nor state found for isolate 25-005882-002
No BioSample nor date nor state found for isolate 25-006120-003
No BioSample nor date nor state found for isolate 25-005756-005
No BioSample nor date nor state found for isolate 25-005756-004
No BioSample nor date nor state found for isolate 25-005754-005
No BioSample nor date nor state found for isolate 25-005754-004
No BioSample nor date nor state found for isolate 25-004339-009
No BioSample nor date nor state found for isolate 25-004493-001
No BioSample nor date nor state found for isolate 25-004493-002
No BioSample nor date nor state found for isolate 25-004498-001
No BioSample nor date nor state found for isolate 25-006183-002
No BioSample nor date nor state found for isolate 25-005870-001
No BioSample nor date nor state found for isolate 25-004423-001
No BioSample nor date nor state found for isolate 25-004415-001
No BioSample nor date nor state found for isolate 25-006403-001
No BioSample nor date nor state found fo

In [5]:
search_collection_date_term("PP752829.1", metadata_genbank)

PP752829.1
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_68277e644120fd291b09516d&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
20-Mar-2024


'Texas|20-Mar-2024'